# tiny-gpt-sae: train on free Colab

This notebook trains the ~20M-parameter GPT from [tiny-gpt-sae](https://github.com/Ammario711/tiny-gpt-sae) on TinyStories using a free **T4 GPU**.

**Free Colab sessions end after a few hours**, so everything important lives on **Google Drive**:
- the tokenized dataset (built once, reused in later sessions)
- the checkpoints (`latest.pt` is written every 250 steps)

**To continue training in a new session, just run all cells again.** Training resumes automatically from the last checkpoint.

Before you start: *Runtime → Change runtime type → T4 GPU*.

## 1. Check the GPU
The T4 has no fast bf16 support, so the code automatically uses **fp16 autocast with a GradScaler**. The scaler multiplies the loss before backprop so small fp16 gradients don't round to zero.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("compute capability:", torch.cuda.get_device_capability(), "(bf16 needs >= 8; T4 is 7.5 -> fp16)")

## 2. Mount Google Drive
All data and checkpoints go to `MyDrive/tiny-gpt-sae/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ROOT = "/content/drive/MyDrive/tiny-gpt-sae"
DATA_DIR = f"{ROOT}/data/tinystories"
RUN_DIR = f"{ROOT}/runs/colab_t4"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
# Cache the Hugging Face download on Drive too, so it isn't re-downloaded each session.
os.environ["HF_HOME"] = f"{ROOT}/hf_cache"

## 3. Get the code and install it
The repo is cloned into the (temporary) Colab disk each session. Only the data and runs live on Drive.

In [ ]:
%cd /content
!test -d tiny-gpt-sae || git clone https://github.com/Ammario711/tiny-gpt-sae
%cd /content/tiny-gpt-sae
!git pull --ff-only
!pip install -q -e ".[data]"

## 4. Prepare TinyStories (first session only)
This downloads TinyStories, tokenizes every story with the GPT-2 BPE tokenizer, and writes `train.bin` / `val.bin` to Drive as flat arrays of token ids (uint16).

It's slow the first time (expect tens of minutes). Later sessions skip it because the files already exist.

In [ ]:
!python scripts/prepare_data.py --out "$DATA_DIR"
!ls -lh "$DATA_DIR"

## 5. Train (resumes automatically)
`configs/colab_t4.yaml` defines the run: 6 layers, d_model 384, context 256, micro-batch 32 × 2 accumulation steps.

- If `RUN_DIR/latest.pt` exists, training continues from it: same weights, optimizer state, LR schedule position and data-sampling RNG.
- Extra snapshots (`ckpt_XXXXXX.pt`) are kept at chosen steps for the later "how do features emerge during training?" experiment.
- Metrics are appended to `log.jsonl`.

**Tip:** after the first ~200 steps, check `time_s` in the log to estimate how many sessions the full run needs.

In [ ]:
!python -m tiny_gpt_sae.train --config configs/colab_t4.yaml --data-dir "$DATA_DIR" --out-dir "$RUN_DIR"

## 6. Plot the loss curves
This reads `log.jsonl`, which collects entries from every session.

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(f"{RUN_DIR}/log.jsonl")]
# If a session died between log writes, steps can repeat after resume; keep the last entry per step.
by_step = {r["step"]: r for r in rows}
steps = sorted(by_step)
plt.figure(figsize=(7, 4))
plt.plot(steps, [by_step[s]["train_loss"] for s in steps], label="train")
if "val_loss" in by_step[steps[0]]:
    plt.plot(steps, [by_step[s].get("val_loss") for s in steps], label="val")
plt.xlabel("optimizer step"); plt.ylabel("cross-entropy loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("tiny-gpt-sae on TinyStories")
plt.show()

## 7. Generate sample stories
Load `latest.pt` and sample a few continuations. `temperature` controls randomness; `top_k` keeps only the k most likely tokens at each step.

In [ ]:
import torch
from tiny_gpt_sae.data import Tokenizer
from tiny_gpt_sae.model import GPT, GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(f"{RUN_DIR}/latest.pt", map_location=device, weights_only=False)
model = GPT(GPTConfig(**ckpt["config"]["model"])).to(device).eval()
model.load_state_dict(ckpt["model"])
tok = Tokenizer()
print(f"checkpoint at step {ckpt['step']}\n")

for prompt in ["Once upon a time", "Lily wanted to", "The little dog"]:
    idx = torch.tensor([tok.encode(prompt)], device=device)
    out = model.generate(idx, max_new_tokens=120, temperature=0.8, top_k=50)
    print(tok.decode(out[0].tolist()).split("<|endoftext|>")[0])
    print("-" * 60)